In [15]:
!pip install mwclient
import mwclient
site = mwclient.Site("en.wikipedia.org")
page = site.Pages['Ethereum']

In [16]:
revs = list(page.revisions())
revs[0]

OrderedDict([('revid', 1282875834),
             ('parentid', 1282478024),
             ('minor', ''),
             ('user', 'Learningman00'),
             ('timestamp',
              time.struct_time(tm_year=2025, tm_mon=3, tm_mday=29, tm_hour=4, tm_min=25, tm_sec=33, tm_wday=5, tm_yday=88, tm_isdst=-1)),
             ('comment',
              '/* The Merge: Transition from Proof-of-Work to Proof-of-Stake */')])

In [17]:
revs = sorted(revs,key=lambda revs:revs['timestamp'])
revs[0]['timestamp']

time.struct_time(tm_year=2014, tm_mon=1, tm_mday=27, tm_hour=1, tm_min=53, tm_sec=45, tm_wday=0, tm_yday=27, tm_isdst=-1)

In [18]:
len(revs)

3832

In [19]:
# first revision was made in this date

import time
df_time = time.strftime("%Y-%m-%d",revs[0]['timestamp'])
df_time

'2014-01-27'

In [20]:
#making  a transformer pipeline for sentiment analysis
from transformers import pipeline
sentiment_pipeline = pipeline("sentiment-analysis")

def sentiment_revs(text):
  sentiment = sentiment_pipeline([text[:250]])[0]
  # return sentiment
  score = sentiment['score']
  if sentiment['label'] == "NEGATIVE":
    score*=-1
  return score
sentiment_revs("hello ayush")

#why we used [0] because it will give the first list of score because huggingface give a list of scores


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


0.999355137348175

In [21]:
#now we will loop over the revs
edits = {}

for rev in revs:
  date = time.strftime("%Y-%m-%d",rev['timestamp'])
  if date not in edits:
    edits[date] = dict(curr_sent=list(),edit_count=0)

  edits[date]['edit_count'] += 1

  comment = rev.get("comment","")
  edits[date]['curr_sent'].append(sentiment_revs(comment))

In [22]:
from statistics import mean

for keys in edits:
  if len(edits[keys]['curr_sent'])> 0:
    edits[keys]['mean_sentiments'] = mean(edits[keys]['curr_sent'])
    edits[keys]["neg_sentiments"] = len([s for s in edits[keys]['curr_sent'] if s < 0]) / len(edits[keys]['curr_sent'])
    edits[keys]["pos_sentiments"] = len([s for s in edits[keys]['curr_sent'] if s > 0]) / len(edits[keys]['curr_sent'])

  else:
    edits[keys]['mean_sentiments'] = 0
    edits[keys]['neg_sentiments'] = 0
    edits[keys]['pos_sentiments'] = 0
  del edits[keys]['curr_sent']
edits

{'2014-01-27': {'edit_count': 1,
  'mean_sentiments': -0.9985105395317078,
  'neg_sentiments': 1.0,
  'pos_sentiments': 0.0},
 '2014-02-01': {'edit_count': 1,
  'mean_sentiments': -0.997276246547699,
  'neg_sentiments': 1.0,
  'pos_sentiments': 0.0},
 '2014-04-06': {'edit_count': 5,
  'mean_sentiments': 0.7909793376922607,
  'neg_sentiments': 0.0,
  'pos_sentiments': 1.0},
 '2014-04-09': {'edit_count': 24,
  'mean_sentiments': 0.6464069038629532,
  'neg_sentiments': 0.08333333333333333,
  'pos_sentiments': 0.9166666666666666},
 '2014-04-10': {'edit_count': 9,
  'mean_sentiments': -0.3615177207522922,
  'neg_sentiments': 0.6666666666666666,
  'pos_sentiments': 0.3333333333333333},
 '2014-04-11': {'edit_count': 10,
  'mean_sentiments': 0.5150125801563263,
  'neg_sentiments': 0.2,
  'pos_sentiments': 0.8},
 '2014-04-12': {'edit_count': 2,
  'mean_sentiments': -0.04391053318977356,
  'neg_sentiments': 0.5,
  'pos_sentiments': 0.5},
 '2014-04-13': {'edit_count': 1,
  'mean_sentiments': 0.74

In [27]:
import pandas as pd
df_sent_eth = pd.DataFrame.from_dict(edits,orient='index')
df_sent_eth.index = pd.to_datetime(df_sent_eth.index)
df_sent_eth

,edit_count,mean_sentiments,neg_sentiments,pos_sentiments
2014-01-27,1,-0.998511,1.000000,0.000000
2014-02-01,1,-0.997276,1.000000,0.000000
2014-04-06,5,0.790979,0.000000,1.000000
2014-04-09,24,0.646407,0.083333,0.916667
2014-04-10,9,-0.361518,0.666667,0.333333
...,...,...,...,...
2025-03-10,1,0.977994,0.000000,1.000000
2025-03-18,7,-0.485668,0.714286,0.285714
2025-03-21,4,-0.466895,0.750000,0.250000
2025-03-26,3,-0.334291,0.666667,0.333333


In [28]:
from datetime import datetime
date = pd.date_range(start='2014-01-27',end=datetime.today())
df_sent_eth = df_sent_eth.reindex(date,fill_value=0)
df_sent_eth

,edit_count,mean_sentiments,neg_sentiments,pos_sentiments
2014-01-27,1,-0.998511,1.0,0.0
2014-01-28,0,0.000000,0.0,0.0
2014-01-29,0,0.000000,0.0,0.0
2014-01-30,0,0.000000,0.0,0.0
2014-01-31,0,0.000000,0.0,0.0
...,...,...,...,...
2025-03-27,0,0.000000,0.0,0.0
2025-03-28,0,0.000000,0.0,0.0
2025-03-29,1,0.996167,0.0,1.0
2025-03-30,0,0.000000,0.0,0.0


In [29]:
df_sent_eth_roll = df_sent_eth.rolling(30, min_periods=30).mean()
df_sent_eth_roll = df_sent_eth_roll.dropna()
df_sent_eth_roll

,edit_count,mean_sentiments,neg_sentiments,pos_sentiments
2014-02-25,0.066667,-0.066526,0.066667,0.000000
2014-02-26,0.033333,-0.033243,0.033333,0.000000
2014-02-27,0.033333,-0.033243,0.033333,0.000000
2014-02-28,0.033333,-0.033243,0.033333,0.000000
2014-03-01,0.033333,-0.033243,0.033333,0.000000
...,...,...,...,...
2025-03-27,0.533333,0.014642,0.071032,0.095635
2025-03-28,0.533333,0.014642,0.071032,0.095635
2025-03-29,0.566667,0.047848,0.071032,0.128968
2025-03-30,0.533333,0.022910,0.071032,0.095635


In [30]:
df_sent_eth_roll.to_csv("df_sent_eth_roll.csv")

In [31]:
len(df_sent_eth_roll)

4053